In [1]:
import os

import dotenv
dotenv.load_dotenv()
from kbel.disambiguators import Disambiguator
from kbel.disambiguators.naive import NaiveDisambiguator
from kbel.disambiguators.similarity import SimilarityDisambiguator
from kbel.disambiguators.llm import LLM_Disambiguator
from kbel.core.mention import Mention
from kbel.core.mention import EntityType
from kbel.knowledge_sources import KnowledgeSource
# import logging
# logging.basicConfig(level=logging.DEBUG)

### Using `naive` disambiguator to link entities from Wikidata

In [2]:
kbel = Disambiguator(strategy_name='naive')
results = kbel.disambiguate(
    mention=Mention(label='rock', text='', entity_type=EntityType.ITEM),
    ks=KnowledgeSource('wikidata', limit=10))
display (*results)

('rock music',
 'popular music genre',
 Item(IRI('http://www.wikidata.org/entity/Q11399')))

### Using `similarity` disambiguator to link entities from Wikidata

In [3]:
kbel = Disambiguator('sim')
results = kbel.disambiguate(
    ks=KnowledgeSource('wikidata-wapi', limit=10),
    mention=Mention(label='Rock', text='Rock is a stone', entity_type=EntityType.ITEM),
    limit=2)
display (*results)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

('stone',
 'rock; building material',
 Item(IRI('http://www.wikidata.org/entity/Q22731')))

('Rock',
 'male given name',
 Item(IRI('http://www.wikidata.org/entity/Q60589667')))

In [4]:
results = kbel.disambiguate(
    ks=KnowledgeSource('wikidata-wapi', limit=10),
    mention=Mention(label='instance of', text='Rock is a stone', entity_type=EntityType.PROPERTY)
)

display(*results)

('instance of',
 'type to which this subject corresponds/belongs. Different from P279 (subclass of); for example: K2 is an instance of mountain; volcano is a subclass of mountain',
 Property(IRI('http://www.wikidata.org/entity/P31'), None))

('subproperty of',
 'all resources related by this property are also related by that property',
 Property(IRI('http://www.wikidata.org/entity/P1647'), None))

('individual of taxon',
 'the taxon of an individual named organism (animal, plant)',
 Property(IRI('http://www.wikidata.org/entity/P10241'), None))

In [8]:
kbel = Disambiguator(strategy_name='sim')
results = kbel.disambiguate(
    ks=KnowledgeSource('dbpedia', limit=10),
    mention=Mention(label='Rock', text='Rock is a naturally occurring solid aggregate of one or more minerals or mineraloids', entity_type=EntityType.ITEM),
    limit=2)
display (*results)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

('Soft rock', '', Item(IRI('http://dbpedia.org/resource/Soft_rock')))

('Rock music', '', Item(IRI('http://dbpedia.org/resource/Rock_music')))

### Using `LLM` disambiguator to link entities from Wikidata

Instantiating LLM Disambiguator with IBM WatsonX's models

In [9]:
from langchain_openai import ChatOpenAI
model = ChatOpenAI(model='gpt-5.2', api_key=os.environ['LLM_API_KEY'])
kbel = Disambiguator('llm', model= model)

In [10]:
results = kbel.disambiguate(
    mention=Mention(label='Rock', text='A rock can be used in construction to mimic the appearance and durability of natural stone.', entity_type=EntityType.ITEM),
    ks=KnowledgeSource('wikidata-wapi', limit=100))

display (*results)

('rock',
 'naturally occurring solid aggregate of one or more minerals or mineraloids',
 Item(IRI('http://www.wikidata.org/entity/Q8063')))